TWCI A4 - Step 1: monthly water-surface area of a reservoir from Sentinel-2 (Google Earth Engine)
Run in Google Colab. Each "%%" block is one cell. Run them in order.
#
THE IDEA (simple): a satellite photo is a grid of small squares (pixels, 10 m x 10 m).
Water reflects light differently from land. NDWI = (green - near-infrared) / (green + near-infrared).
If NDWI > 0 the square is water. Count water squares x 100 m2 = water area. Do it for every month.

### 1. Log in to Earth Engine (a window opens, press Allow)

In [ ]:
import ee, pandas as pd
ee.Authenticate(auth_mode="notebook")
ee.Initialize(project="twci26")

### 2. SETTINGS - the only things you change

In [ ]:
# Draw a rectangle around the reservoir in the Earth Engine map, copy 4 corners as [west, south, east, north]
NAME = "Iriklinskoye"            # for Shardara write "Shardara"
WEST, SOUTH, EAST, NORTH = 58.58, 51.78, 59.28, 51.93   # Iriklinskoye: your 2 corner points + safety margin
START_YEAR, END_YEAR = 2015, 2024
NDWI_THRESHOLD = 0.0             # water if NDWI > this number (fixed BEFORE looking at results)
MIN_VALID_SHARE = 0.60           # month is kept only if >= 60% of the rectangle has clear pixels
assert WEST is not None, "Put the 4 numbers in WEST, SOUTH, EAST, NORTH first"
aoi = ee.Geometry.Rectangle([WEST, SOUTH, EAST, NORTH])

### 3. Sentinel-2 images with clouds removed (SCL = scene classification layer)

In [ ]:
def clear_and_ndwi(img):
    scl = img.select("SCL")
    # bad classes: 3 cloud shadow, 8 and 9 cloud, 10 thin cirrus, 11 snow/ice
    clear = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("ndwi")
    return ndwi.updateMask(clear).addBands(clear.rename("clear"))

def month_stats(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(aoi).filterDate(start, end)
           .map(clear_and_ndwi))
    n_images = col.size()
    ndwi_med = col.select("ndwi").median()                       # middle value of the month
    clear_share_img = col.select("clear").mean().gt(0.0)          # was this square ever clear?
    water = ndwi_med.gt(NDWI_THRESHOLD).selfMask()
    px = ee.Image.pixelArea()                                     # area of each square, m2
    water_m2 = px.updateMask(water).reduceRegion(ee.Reducer.sum(), aoi, 10, maxPixels=1e10).get("area")
    valid_m2 = px.updateMask(clear_share_img).reduceRegion(ee.Reducer.sum(), aoi, 10, maxPixels=1e10).get("area")
    total_m2 = px.reduceRegion(ee.Reducer.sum(), aoi, 10, maxPixels=1e10).get("area")
    return ee.Feature(None, {"year": year, "month": month, "n_images": n_images,
                             "water_m2": water_m2, "valid_m2": valid_m2, "total_m2": total_m2})

### 3b. CHECK: does the rectangle cover the whole reservoir? (open the link, you should see the lake in blue fully inside)

In [ ]:
col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(aoi)
       .filterDate("2020-07-01", "2020-08-01").map(clear_and_ndwi))
water = col.select("ndwi").median().gt(NDWI_THRESHOLD)
img = water.visualize(min=0, max=1, palette=["white", "blue"])
print(img.getThumbURL({"region": aoi, "dimensions": 900, "format": "png"}))


### 4. Run for every month, one by one (takes about 10 minutes; one line appears per finished year, wait)

In [ ]:
import time
rows = []
for y in range(START_YEAR, END_YEAR + 1):
    for m in range(1, 13):
        # first a quick, cheap question: are there ANY photos this month? (early months, 2015-2017, often have none)
        start = ee.Date.fromYMD(y, m, 1)
        n_img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(aoi)
                 .filterDate(start, start.advance(1, "month")).size().getInfo())
        if n_img == 0:
            rows.append({"year": y, "month": m, "n_images": 0, "water_m2": None, "valid_m2": None, "total_m2": None})
            continue                                    # no photos = Blind Spot Register, NOT guessed
        for attempt in range(6):                      # if Earth Engine says "too busy", wait and try again
            try:
                rows.append(month_stats(y, m).getInfo()["properties"])
                break
            except Exception as e:
                time.sleep(8 * (attempt + 1))
        else:
            print("FAILED:", y, m)
    print(y, "done")                                    # one line per year = progress
df = pd.DataFrame(rows)
df["water_km2"] = df["water_m2"].fillna(0) / 1e6
df["valid_share"] = df["valid_m2"].fillna(0) / df["total_m2"]
df["usable"] = (df["n_images"] > 0) & (df["valid_share"] >= MIN_VALID_SHARE)
df.loc[~df["usable"], "water_km2"] = float("nan")   # unusable months are NOT guessed - they go to the Blind Spot Register
df["date"] = pd.to_datetime(dict(year=df.year, month=df.month, day=1))
df = df.sort_values("date")
print(df[["date", "n_images", "valid_share", "water_km2", "usable"]].to_string(index=False))
print("Usable months:", int(df.usable.sum()), "of", len(df))

### 5. Save two files

In [ ]:
df.to_csv(f"{NAME}_area_monthly.csv", index=False)
df.loc[~df.usable, ["date", "n_images", "valid_share"]].to_csv(f"{NAME}_blind_spot_register.csv", index=False)
from google.colab import files
files.download(f"{NAME}_area_monthly.csv")
files.download(f"{NAME}_blind_spot_register.csv")

### 6. Quick picture (your first check: does it look like a reservoir that fills and empties?)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3.5))
plt.plot(df["date"], df["water_km2"], marker="o", linewidth=1)
plt.ylabel("water area, km2"); plt.title(f"{NAME}: monthly water area (gaps = unusable months)")
plt.tight_layout(); plt.show()